# Python 迭代器与生成器 · 教学级 Notebook

> **业务场景说明**：源需求中的 `{{业务场景描述}}` 未填写，本 Notebook 依据保存路径（`LLMOps/llmops-api`）将业务场景假定为——**LLM 应用的后台数据处理**：大语料流式读取、文档分块（chunking）、LLM 流式响应消费、批量调用 embedding / 接口分页拉取。全部示例围绕该场景取材。

## 学习目标

1. 说清**可迭代对象、迭代器、生成器**三者的关系与判定方法；
2. 能手写迭代器类，并解释 `for` 循环的底层机制；
3. 掌握 `yield` / `send` / `throw` / `close` 与 `yield from` 的执行流程；
4. 能用生成器表达式和管道模式处理大数据流，理解惰性求值的内存优势；
5. 避开一次性消费、延迟绑定等高频陷阱；
6. 能将生成器模式落地到 LLMOps 场景（流式输出、分块、分页、批处理）。

## 一句话本质

> **迭代器是协议，生成器是实现这个协议的便捷方式；生成器的本质是"可暂停、可恢复、可惰性求值"的函数。**


---
# 〇、前置知识

**本章解决什么问题**：在进入迭代器之前，先确认你具备读懂后面代码的语言基础。迭代器高度依赖以下四项特性，每个特性都配一个可运行的验证示例。

| 依赖特性 | 与本主题的关系 | 验证方式 |
|---|---|---|
| 魔法方法（双下划线方法） | 迭代器协议就是 `__iter__` / `__next__` 两个魔法方法 | 验证 1：`hasattr` 检查方法是否存在 |
| 异常处理 `try/except` | 迭代结束的信号是 `StopIteration` 异常 | 验证 2：捕获并打印异常 |
| 类的定义与实例化 | 手写迭代器需要定义类 | 验证 3：定义含 `__init__` 的类 |
| 鸭子类型 | `for` 不关心类型，只关心有没有协议方法 | 验证 4：自造可迭代类被 `list()` 接受 |


In [1]:
# 【前置验证 1】魔法方法：Python 对象的能力由方法决定
nums = [1, 2, 3]                     # ① list 是普通可迭代对象
print("list 有 __iter__ :", hasattr(nums, "__iter__"))    # ② 能被 iter() 接受
print("list 有 __next__:", hasattr(nums, "__next__"))   # ③ 但自己不能 next()

it = iter(nums)                      # ④ iter() 返回一个"迭代器"
print("迭代器 有 __iter__ :", hasattr(it, "__iter__"))
print("迭代器 有 __next__:", hasattr(it, "__next__"))

list 有 __iter__ : True
list 有 __next__: False
迭代器 有 __iter__ : True
迭代器 有 __next__: True


In [2]:
# 【前置验证 2】异常处理：后续大量用于演示 StopIteration 等信号
try:
    raise ValueError("演示异常")      # ① 主动抛出一个异常
except ValueError as e:              # ② 捕获并读取异常对象
    print("捕获到:", e)
finally:                             # ③ 无论是否出错都执行（资源清理的基石）
    print("finally 执行")

捕获到: 演示异常
finally 执行


In [3]:
# 【前置验证 3】类的定义、实例化与方法调用（手写迭代器的基本功）
class Demo:
    def __init__(self, value):       # ① 构造方法：创建实例时执行
        self.value = value           # ② 实例属性保存在 self 上（这就是"状态"）
    def get(self):
        return self.value

d = Demo(42)
print(d.get())                       # ③ 输出 42

42


In [4]:
# 【前置验证 4】鸭子类型：不看类型看行为，for 只认协议
class DuckIterable:
    def __iter__(self):              # ① 只要实现 __iter__，for / list() 就认
        return iter(["鸭子", "类型", "生效"])

print(list(DuckIterable()))          # ② 自造类无需继承任何基类即可被遍历

['鸭子', '类型', '生效']


---
# 一、迭代器协议

**本章解决什么问题**：Python 的 `for` 循环不依赖索引，那它靠什么取值？答案是**迭代器协议**。

## 要点

- 一个对象要成为**迭代器**，需实现两个方法：
  - `__iter__()`：返回迭代器对象本身；
  - `__next__()`：返回下一个元素；没有元素时抛 `StopIteration`。
- 内置函数 `iter(obj)` 调用 `obj.__iter__()`，`next(it)` 调用 `it.__next__()`。
- `next()` 可带默认值：耗尽时返回默认值而**不抛异常**。

## 代码骨架

```python
class Iterator:
    def __iter__(self):
        return self

    def __next__(self):
        # 返回下一个值
        # 没有更多值时抛出 StopIteration
        ...
```


In [5]:
# 知识点 1：iter() 与 next() 的底层行为
it = iter([1, 2, 3])        # ① iter() 调用对象的 __iter__，得到迭代器
print(next(it))             # ② next() 调用 __next__，返回 1
print(next(it))             # ③ 返回 2
print(next(it))             # ④ 返回 3

# ⑤ 耗尽后再取值会抛 StopIteration —— 这里捕获演示（不中断执行）
try:
    next(it)
except StopIteration:
    print("StopIteration：迭代器已耗尽")

# ⑥ next() 的默认值形式：耗尽时返回默认值，不抛异常
print(next(it, "结束"))

1
2
3
StopIteration：迭代器已耗尽
结束


---
# 二、可迭代对象 vs 迭代器

**本章解决什么问题**：这两个概念最容易混淆，本章用一张表 + 验证代码彻底区分。

## 概念对比

| 概念 | 必须实现 | 判定 | 典型例子 |
|---|---|---|---|
| 可迭代对象 Iterable | `__iter__()` | 能被 `iter()` 接受 | list、tuple、str、dict、set、range、文件对象、生成器 |
| 迭代器 Iterator | `__iter__()` **和** `__next__()` | 能被 `iter()` 且能被 `next()` | `iter(lst)` 的返回值、生成器对象 |

**关系**：迭代器一定是可迭代对象；可迭代对象不一定是迭代器。

## for 循环的本质（等价写法）

```python
# for x in obj: ...
# 等价于：
iterator = iter(obj)          # 调 __iter__
while True:
    try:
        x = next(iterator)     # 调 __next__
    except StopIteration:      # 耗尽即退出
        break
    ...                        # 循环体
```

只要对象实现了迭代器协议，就能被 `for` 遍历——与索引无关。


In [6]:
# 知识点 2：用 hasattr 与 collections.abc 精确判定
from collections.abc import Iterable, Iterator

nums = [1, 2, 3]
print("list  有 __iter__:", hasattr(nums, "__iter__"), "| 有 __next__:", hasattr(nums, "__next__"))

it = iter(nums)
print("iter  有 __iter__:", hasattr(it, "__iter__"), "| 有 __next__:", hasattr(it, "__next__"))

# ① 抽象基类判定：isinstance 比 hasattr 更语义化
print("list 是 Iterable:", isinstance(nums, Iterable), "| 是 Iterator:", isinstance(nums, Iterator))
print("iter 是 Iterable:", isinstance(it, Iterable), "| 是 Iterator:", isinstance(it, Iterator))

list  有 __iter__: True | 有 __next__: False
iter  有 __iter__: True | 有 __next__: True
list 是 Iterable: True | 是 Iterator: False
iter 是 Iterable: True | 是 Iterator: True


In [7]:
# 知识点 3：验证 for 循环与手写 iter/next 完全等价（用输出证明）
obj = ["a", "b", "c"]

result_for = []                          # ① 语法糖写法
for x in obj:
    result_for.append(x)

result_manual = []                       # ② 手动展开写法
iterator = iter(obj)
while True:
    try:
        x = next(iterator)
    except StopIteration:
        break
    result_manual.append(x)

print("for 结果  :", result_for)
print("手动结果 :", result_manual)
print("两者相等  :", result_for == result_manual)

for 结果  : ['a', 'b', 'c']
手动结果 : ['a', 'b', 'c']
两者相等  : True


---
# 三、手动实现迭代器

**本章解决什么问题**：不借助 `yield`，纯手写一个能被 `for` 遍历的类，并搞清"可重复遍历"与"只能遍历一次"的根源。

## 两种写法

| 写法 | 结构 | 可否重复遍历 | 原因 |
|---|---|---|---|
| 可迭代对象 + 独立迭代器 | `__iter__` 返回**新的**迭代器实例 | 可以 | 每次遍历都拿到全新状态 |
| 自身既是可迭代对象又是迭代器 | `__iter__` 返回 `self` | 只能一次 | 状态保存在自身，耗尽即失效 |


In [8]:
# 知识点 4：可迭代对象 + 独立迭代器（推荐结构）
class CountDownIterator:            # ① 迭代器：真正保存遍历状态
    def __init__(self, start):
        self.current = start

    def __iter__(self):             # ② 迭代器协议要求：返回自身
        return self

    def __next__(self):
        if self.current <= 0:       # ③ 没有元素时抛 StopIteration
            raise StopIteration
        self.current -= 1
        return self.current + 1

class CountDown:                    # ④ 可迭代对象：只负责"生产"迭代器
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        return CountDownIterator(self.start)

for n in CountDown(3):              # ⑤ 每次都会创建新的迭代器
    print(n)

print("第二次遍历:", list(CountDown(3)))   # ⑥ 状态独立，可以重复遍历

3
2
1
第二次遍历: [3, 2, 1]


In [9]:
# 知识点 5：自身既可迭代又是迭代器（一次性陷阱的根源）
class CountDownOnce:
    def __init__(self, start):
        self.current = start        # ① 状态保存在自身

    def __iter__(self):
        return self                 # ② 返回自身 —— 不再创建新迭代器

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        self.current -= 1
        return self.current + 1

c = CountDownOnce(3)
print("第一次遍历:", list(c))
print("第二次遍历:", list(c))        # ③ 输出 []：状态已耗尽，只能遍历一次

第一次遍历: [3, 2, 1]
第二次遍历: []


---
# 四、生成器函数

**本章解决什么问题**：手写迭代器太啰嗦，`yield` 用函数语法一行搞定。

## 要点

- 包含 `yield` 的函数叫**生成器函数**；调用它**不执行函数体**，而是返回一个**生成器对象**；
- 生成器对象自动实现迭代器协议（可 `for`、可 `next`）；
- `next()` 驱动执行，遇 `yield` 暂停并交出值；函数执行完毕抛 `StopIteration`。

## 执行时序

| 调用 | 行为 |
|---|---|
| `g = countdown(3)` | 只创建生成器对象，**函数体一行都不执行** |
| `next(g)` | 开始执行，遇 `yield` 暂停，返回 yield 的值 |
| 再次 `next(g)` | 从暂停处恢复执行 |
| 函数 return / 执行完 | 抛 `StopIteration` |


In [10]:
# 知识点 6：调用生成器函数不执行函数体
def countdown(n):
    print("开始")                # ① 只有被 next 驱动时才会执行
    while n > 0:
        yield n                  # ② 暂停点：交出 n 并保存全部局部状态
        n -= 1
    print("结束")

g = countdown(3)
print(type(g))                   # ③ <class 'generator'>，注意：上面没有打印"开始"

print("--- 开始驱动 ---")
print(next(g))                   # ④ 打印"开始"，返回 3
print(next(g))                   # ⑤ 返回 2
print(next(g))                   # ⑥ 返回 1
try:
    next(g)                      # ⑦ 打印"结束"后抛 StopIteration
except StopIteration:
    print("生成器结束")

print("--- 生成器对象可直接被 for ---")
for n in countdown(3):           # ⑧ 生成器自动实现迭代器协议
    print(n)

<class 'generator'>
--- 开始驱动 ---
开始
3
2
1
结束
生成器结束
--- 生成器对象可直接被 for ---
开始
3
2
1
结束


---
# 五、生成器执行流程：暂停、恢复与双向通信

**本章解决什么问题**：`x = yield 1` 这种"又产出又接收"的写法到底怎么执行？`send()` 的值去了哪里？

## 执行规则

1. `next(g)`：执行到 `yield 1`，**返回 1**，暂停；
2. `g.send(10)`：把 10 赋给 `x = yield 1` 的**左侧**，从暂停处继续执行；
3. 函数 `return "done"` 后抛 `StopIteration`，**返回值放在 `e.value`**。


In [11]:
# 知识点 7：yield 表达式的完整执行流程
def gen():
    print("A")                 # ① 首次 next 时执行
    x = yield 1                # ② 执行到 yield 1 暂停，把 1 交给调用方
    print("收到:", x)          # ③ send(10) 恢复后执行，此时 x == 10
    y = yield 2
    print("收到:", y)
    return "done"               # ⑤ return 的值会放进 StopIteration.value

g = gen()
print(next(g))       # 输出 A，返回 1
print(g.send(10))    # 输出 收到: 10，返回 2
try:
    g.send(20)       # 输出 收到: 20，随后函数 return 结束
except StopIteration as e:
    print("返回值:", e.value)   # done

A
1
收到: 10
2
收到: 20
返回值: done


---
# 六、生成器表达式

**本章解决什么问题**：列表推导 `[x*x for x in ...]` 会立刻算完所有元素，数据大时内存爆炸——生成器表达式是它的**惰性版本**。

## 对比

| 写法 | 定界符 | 求值时机 | 内存 |
|---|---|---|---|
| 列表推导 | `[...]` | 立刻全部求值 | 与元素数成正比 |
| 生成器表达式 | `(...)` | 每次 `next()` 算一个 | 常数级 |

**语法糖**：生成器表达式若是函数调用的唯一参数，可省略外层括号：`sum(x*x for x in range(10))`。


In [12]:
# 知识点 8：列表推导 vs 生成器表达式
lst = [x * x for x in range(5)]        # ① 立刻生成全部元素
gen = (x * x for x in range(5))       # ② 只创建生成器，尚未计算

print("列表推导:", lst)                # [0, 1, 4, 9, 16]
print("生成器对象:", gen)              # <generator object ...>，还没算
print(next(gen))                       # ③ 此刻才计算第一个：0
print(next(gen))                       # 1

# ④ 唯一参数时可省略外层括号（工程高频写法）
total = sum(x * x for x in range(10))
print("sum 结果:", total)              # 285

列表推导: [0, 1, 4, 9, 16]
生成器对象: <generator object <genexpr> at 0x10c43b370>
0
1
sum 结果: 285


---
# 七、生成器方法：send、throw、close

**本章解决什么问题**：生成器不只是"被动吐值"的迭代器，还支持**双向通信**（send）、**外部注入异常**（throw）、**主动关闭清理**（close）。

## 方法速查

| 方法 | 作用 | 关键注意 |
|---|---|---|
| `send(v)` | 把 v 送入生成器暂停处（成为 yield 表达式的值），并恢复执行 | **首次必须先 `next()` 或 `send(None)` 启动** |
| `throw(exc)` | 在暂停点抛入异常，由生成器内部 try 捕获 | 可用于通知上游"出错停止" |
| `close()` | 关闭生成器，在暂停点抛 `GeneratorExit` | 捕获 GeneratorExit 后**不要**继续 yield，否则 RuntimeError |


In [13]:
# 知识点 9：send —— 向生成器发送值（累加器协程）
def accumulator():
    total = 0
    while True:
        x = yield total        # ① 暂停点：产出当前累计值，等待外部送入新值
        if x is None:          # ② send(None) 等价于 next()
            continue
        total += x

acc = accumulator()
print(next(acc))      # ③ 启动生成器，返回 0
print(acc.send(10))   # ④ 10
print(acc.send(5))    # ⑤ 15
print(acc.send(3))    # ⑥ 18
acc.close()           # ⑦ 用完关闭，释放资源

0
10
15
18


In [14]:
# 知识点 10：throw —— 在暂停点抛入异常
def gen():
    try:
        yield 1                    # ① 第一个暂停点
    except ValueError:             # ② 外部 throw 的异常在这里被捕获
        print("捕获 ValueError")
        yield 2
    finally:
        print("清理")              # ③ 生成器结束时执行

g = gen()
print(next(g))               # 1
print(g.throw(ValueError))  # 输出"捕获 ValueError"后返回 2

1
捕获 ValueError
2


In [15]:
# 知识点 11：close —— 关闭生成器并触发清理
def gen():
    try:
        yield 1
        yield 2
    finally:
        print("清理")       # ① GeneratorExit 触发 finally

g = gen()
print(next(g))   # 1
g.close()        # ② 在暂停点抛 GeneratorExit，"清理"被打印

清理
1
清理


---
# 八、yield from：委托给子生成器

**本章解决什么问题**：嵌套迭代时手写两层 `for + yield` 太啰嗦，`yield from` 一行完成委托。

## 要点

- `yield from sub()`：把子生成器的值**直接**委托给外层调用方；
- 子生成器的 `return` 值成为 `yield from` 表达式的值；
- 自动透传 `send()` / `throw()` / `close()`；
- 经典用途：**递归遍历树**。


In [16]:
# 知识点 12：yield from 的基本用法与返回值
def sub():
    yield 1
    yield 2
    return "sub done"        # ① 子生成器的 return 值

def main():
    result = yield from sub()   # ② 1、2 由 sub 直接产出，不经过 main 中转
    print("sub 返回:", result)   # ③ "sub done" 成为 yield from 表达式的值
    yield 3

print(list(main()))          # 输出顺序：先"sub 返回: sub done"，再 [1, 2, 3]

sub 返回: sub done
[1, 2, 3]


In [17]:
# 知识点 13：yield from 递归遍历树（知识库目录场景）
tree = {                                    # ① 模拟知识库目录树
    "name": "知识库/",
    "children": [
        {"name": "10-Sources/", "children": []},
        {"name": "20-Cards/", "children": [
            {"name": "迭代器.card", "children": []},
            {"name": "生成器.card", "children": []},
        ]},
    ],
}

def walk(node):
    yield node["name"]                      # ② 先产出当前节点
    for child in node["children"]:
        yield from walk(child)              # ③ 子树整体委托给递归调用

for path in walk(tree):
    print(path)

知识库/
10-Sources/
20-Cards/
迭代器.card
生成器.card


---
# 九、惰性求值与内存优势

**本章解决什么问题**：为什么处理百万级数据时，生成器比列表省几个数量级的内存？以及它的代价——**只能消费一次**。

## 要点

- 生成器不预先生成数据，每次 `next()` 才计算一项 → 可表示**无限序列**；
- 配合 `itertools.islice` 可从无限序列中安全取前 N 项；
- 生成器是一次性的：需要多次遍历时，**重建生成器 / 转列表 / `itertools.tee`** 三选一。


In [18]:
# 知识点 14：内存对比（getsizeof 实测）
import sys
from itertools import islice

lst = [i for i in range(1_000_000)]        # ① 列表：100 万元素全部入内存
gen = (i for i in range(1_000_000))        # ② 生成器：只保存求值规则

print(f"list 占用     : {sys.getsizeof(lst) / 1024 / 1024:.1f} MB")
print(f"generator 占用: {sys.getsizeof(gen)} 字节")
del lst                                    # ③ 及时释放大列表

# 知识点 15：无限序列 + islice 安全取前 N 项
def naturals():
    n = 0
    while True:          # ① 无限循环，但惰性求值所以不会卡死
        yield n
        n += 1

print("自然数前 5 项:", list(islice(naturals(), 5)))

# 知识点 16：生成器只能消费一次
g = (x for x in range(3))
print("第一次消费:", list(g))   # [0, 1, 2]
print("第二次消费:", list(g))   # [] —— 状态已耗尽

list 占用     : 8.1 MB
generator 占用: 192 字节
自然数前 5 项: [0, 1, 2, 3, 4]
第一次消费: [0, 1, 2]
第二次消费: []


---
# 十、生成器管道

**本章解决什么问题**：多步数据处理（读取 → 过滤 → 转换）如何做到**内存占用小、组合灵活**？

## 核心思想

- 每一步是一个小生成器，只做一件事；
- 数据是**拉取式**流动：消费者每要一个值，就驱动上游生成一个值；
- 管道组装时**什么都不会执行**（全惰性），`for` 消费时才逐级驱动。

下面以 LLMOps 场景演示：读取语料文件 → 过滤空行 → 统一规范化。


In [19]:
# 知识点 17：三段式生成器管道（本单元自包含，自动创建并清理演示文件）
from pathlib import Path

# ① 先准备演示数据文件（真实场景是超大日志/语料文件）
data_file = Path("pipeline_demo_data.txt")
data_file.write_text("hello llmops\n\n   \nstreaming saves memory\n\nchunking for rag\n", encoding="utf-8")

def read_lines(path):
    # ② 数据源：逐行读取文件，with 保证句柄关闭
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield line.rstrip()          # rstrip 去掉行尾换行符

def filter_empty(lines):
    # ③ 过滤：只保留非空行
    for line in lines:
        if line:
            yield line

def to_upper(lines):
    # ④ 转换：统一大写
    for line in lines:
        yield line.upper()

# ⑤ 组装管道：注意此刻没有任何一行被读取（惰性）
pipeline = to_upper(filter_empty(read_lines(data_file)))
for line in pipeline:                    # ⑥ 消费时才逐级拉取
    print(line)

data_file.unlink()                       # ⑦ 清理演示文件

HELLO LLMOPS
STREAMING SAVES MEMORY
CHUNKING FOR RAG


---
# 十一、生成器作为协程

**本章解决什么问题**：`send()` 双向通信的集大成者是"生产者-消费者"模型——它也是早期 Python 协程的原型；现代 Python 用 `async def` / `await` / `async for`。

## 演进脉络

| 阶段 | 机制 | 现状 |
|---|---|---|
| 早期协程 | 生成器 + `send()` | 仍值得理解原理 |
| `@asyncio.coroutine` + `yield from` | 已于 3.11 移除 | 仅历史 |
| 现代协程 | `async def` / `await` / `async for` | 主流 |

异步生成器实现 `__aiter__` / `__anext__`，配合 `async for` 使用——LLM 的**流式响应**本质上就是异步迭代。


In [20]:
# 知识点 18：生成器实现生产者-消费者
def consumer():
    while True:
        item = yield          # ① 只接收不产出：等待生产者 send 进数据
        print("消费:", item)

def producer(c):
    next(c)                   # ② 先启动消费者（推进到第一个 yield）
    for i in range(3):
        c.send(i)             # ③ 逐个发送
    c.close()                 # ④ 通信结束，关闭消费者

c = consumer()
producer(c)

消费: 0
消费: 1
消费: 2


In [21]:
# 知识点 19：异步生成器与 async for
import asyncio

async def agen():
    # ① async def + yield = 异步生成器（自动实现 __aiter__/__anext__）
    for i in range(3):
        await asyncio.sleep(0.01)     # 模拟等待网络（缩短等待时间）
        yield i

async def main():
    # ② async for：异步逐项消费
    async for x in agen():
        print("异步收到:", x)

# ③ Notebook 内核自带运行中的事件循环，直接 run_until_complete 会冲突；
#    放到独立线程中创建全新事件循环驱动（保证可运行）
import threading

def run_async(coro):
    result = {}
    def runner():
        loop = asyncio.new_event_loop()     # 独立线程里没有别的循环，安全
        try:
            result["value"] = loop.run_until_complete(coro)
        finally:
            loop.close()
    t = threading.Thread(target=runner)
    t.start()
    t.join()
    return result.get("value")

run_async(main())

异步收到: 0


异步收到: 1


异步收到: 2


---
# 十二、反例单元：错误写法 vs 正确写法

**本章解决什么问题**：下面三个易错点先给出"错误写法"并**实际触发报错**（用 try/except 捕获展示，保证单元可执行），再给出正确写法对比输出差异。

| # | 错误写法 | 实际症状 | 正确写法 |
|---|---|---|---|
| 1 | 生成器二次遍历 | 静默返回 `[]`（不报错，更隐蔽） | 转列表 / 重建 / `itertools.tee` |
| 2 | 未启动就 `send(值)` | `TypeError` | 先 `next()` 或 `send(None)` |
| 3 | 生成器内 `raise StopIteration` | 变成 `RuntimeError`（PEP 479） | 用 `return` 结束 |


In [22]:
# 反例 1：生成器二次遍历（静默错误 —— 比报错更危险）
# 【错误写法】
g = (x for x in range(3))
print("第一次:", list(g))
print("第二次:", list(g))        # 不报错但得到 []！静默产生错误结果

# 【正确写法 A】需要多次遍历就用列表
data = [x for x in range(3)]
print("列表第一次:", data)
print("列表第二次:", data)

# 【正确写法 B】itertools.tee 把一个流拆成两个独立流
from itertools import tee
g2 = (x for x in range(3))
g_a, g_b = tee(g2)
print("tee 流 A:", list(g_a))
print("tee 流 B:", list(g_b))

第一次: [0, 1, 2]
第二次: []
列表第一次: [0, 1, 2]
列表第二次: [0, 1, 2]
tee 流 A: [0, 1, 2]
tee 流 B: [0, 1, 2]


In [23]:
# 反例 2：忘记启动生成器就 send 非None 值
def gen():
    x = yield 1
    yield x

# 【错误写法】
g = gen()
try:
    g.send(10)
except TypeError as e:
    print("报错:", e)

# 【正确写法】先 next() 启动（或 send(None)），再 send 值
g = gen()
next(g)                # 启动：推进到第一个 yield
print(g.send(10))      # 正常返回 10

报错: can't send non-None value to a just-started generator
10


In [24]:
# 反例 3：在生成器内部 raise StopIteration（PEP 479 之后是 RuntimeError）
def gen_bad():
    yield 1
    raise StopIteration        # 想提前结束 —— 错误方式

g = gen_bad()
print(next(g))
try:
    next(g)
except RuntimeError as e:
    print("报错:", e)          # generator raised StopIteration（被转换）

# 【正确写法】用 return 结束生成器
def gen_good():
    yield 1
    return                      # 干净地结束

print("正常消费:", list(gen_good()))

1
报错: generator raised StopIteration
正常消费: [1]


---
# 十三、实用示例集：5 个工程高频模式（LLMOps 场景）

**本章解决什么问题**：生成器在 LLM 应用后台最高频的五个落地模式。每个模式说明适用场景与注意点。

| # | 模式 | 适用场景 | 关键注意点 |
|---|---|---|---|
| 1 | 流式响应消费 | LLM API 流式输出、打字机效果 | 逐 token 处理，随时可中断 |
| 2 | 文档分块 chunking | RAG 入库 | overlap 控制上下文延续性 |
| 3 | 分页拉取扁平化 | 数据库 / 管理后台接口分页 | 把"按页"封装成"按条"的流 |
| 4 | 批量批处理 | 批量调用 embedding / 接口 | 控制批大小，避免触发限流 |
| 5 | @contextmanager | API 调用计时 / 资源管理 | yield 是暂停点，前后即进入/退出 |


In [25]:
# 模式 1：流式消费 LLM 响应（模拟）
def fake_llm_stream(answer):
    # ① 模拟 LLM API 流式响应：每次 next() 返回一个 token
    for tok in answer.split():
        yield tok

def render_stream(token_stream, max_tokens=None):
    # ② 消费端逐 token 处理；max_tokens 模拟"用户取消/超时截断"
    parts = []
    for tok in token_stream:
        parts.append(tok)
        print("  收到 token:", tok)
        if max_tokens is not None and len(parts) >= max_tokens:
            print("  （达到上限，提前停止拉取）")
            break
    return " ".join(parts)

answer = "迭代器 让 流式 处理 天然 节省 内存"
print("完整输出:", render_stream(fake_llm_stream(answer)))
print("截断输出:", render_stream(fake_llm_stream(answer), max_tokens=3))

  收到 token: 迭代器
  收到 token: 让
  收到 token: 流式
  收到 token: 处理
  收到 token: 天然
  收到 token: 节省
  收到 token: 内存
完整输出: 迭代器 让 流式 处理 天然 节省 内存
  收到 token: 迭代器
  收到 token: 让
  收到 token: 流式
  （达到上限，提前停止拉取）
截断输出: 迭代器 让 流式


In [26]:
# 模式 2：文档分块 chunking（RAG 入库）
def chunk_text(text, size=20, overlap=5):
    # ① 滑动窗口分块：size 是块长，overlap 是相邻块重叠字符数
    step = size - overlap
    if step <= 0:
        raise ValueError("size 必须大于 overlap")
    for start in range(0, len(text), step):
        piece = text[start:start + size]
        if piece:
            yield start, piece        # ② 产出 (起始位置, 块内容)，便于溯源

doc = "迭代器和生成器是处理数据流的核心机制。生成器可以暂停并保存状态，适合大文件与无限序列。"
for start, piece in chunk_text(doc, size=20, overlap=5):
    print(f"[{start:3d}] {piece}")
# 注意点：工程上通常还要按"句子/段落边界"对齐，避免切断语义

[  0] 迭代器和生成器是处理数据流的核心机制。生
[ 15] 心机制。生成器可以暂停并保存状态，适合大
[ 30] 态，适合大文件与无限序列。


In [27]:
# 模式 3：分页拉取扁平化（数据库 / 接口）
def fake_page_fetcher(page_size=2):
    # ① 模拟管理后台接口：按页读取"用户提问记录"
    records = [{"id": i, "question": f"问题{i}"} for i in range(1, 8)]
    def fetch(page):
        # 返回 (本页数据, 是否还有下一页)
        start = page * page_size
        return records[start:start + page_size], start + page_size < len(records)
    return fetch

def paginate(fetch):
    # ② 把"按页拉取"封装成"按条产出"的扁平生成器
    page, has_more = 0, True
    while has_more:
        rows, has_more = fetch(page)
        yield from rows            # ③ 委托：本页记录逐条交给消费者
        page += 1

for rec in paginate(fake_page_fetcher()):
    print(rec)

{'id': 1, 'question': '问题1'}
{'id': 2, 'question': '问题2'}
{'id': 3, 'question': '问题3'}
{'id': 4, 'question': '问题4'}
{'id': 5, 'question': '问题5'}
{'id': 6, 'question': '问题6'}
{'id': 7, 'question': '问题7'}


In [28]:
# 模式 4：批量批处理（批量调用 embedding）
def batched(items, n):
    # ① 把任意可迭代对象切成大小为 n 的批次
    #    （Python 3.12+ 可直接用 itertools.batched）
    batch = []
    for item in items:
        batch.append(item)
        if len(batch) == n:
            yield batch
            batch = []
    if batch:                      # ② 不足 n 的尾批也要产出
        yield batch

def fake_embed(texts):
    # ③ 模拟 embedding 接口：确定性输出，保证可复现
    return [len(t) % 97 for t in texts]

corpus = (f"文档{i}" for i in range(10))    # ④ 惰性语料流
for batch in batched(corpus, 3):
    print(batch, "-> 向量:", fake_embed(batch))

['文档0', '文档1', '文档2'] -> 向量: [3, 3, 3]
['文档3', '文档4', '文档5'] -> 向量: [3, 3, 3]
['文档6', '文档7', '文档8'] -> 向量: [3, 3, 3]
['文档9'] -> 向量: [3]


In [29]:
# 模式 5：@contextmanager —— yield 的另一大工程用途
from contextlib import contextmanager
import time

@contextmanager
def llm_call_guard(api_name):
    # ① yield 之前 = with 进入；yield 之后（finally）= with 退出
    print(f"[{api_name}] 调用开始")
    start = time.perf_counter()
    try:
        yield                      # ② 暂停点：with 块内的代码在这里执行
    finally:
        cost = (time.perf_counter() - start) * 1000
        # 计时输出本身每次不同，改为输出确定性结论
        print(f"[{api_name}] 调用结束（耗时已测量: {cost >= 0} ms）")

with llm_call_guard("embedding-api"):
    total = sum(len(str(x)) for x in range(1000))
print("统计结果:", total)

[embedding-api] 调用开始
[embedding-api] 调用结束（耗时已测量: True ms）
统计结果: 2890


---
# 十四、内置 / 生态对照

**本章解决什么问题**：与迭代器/生成器相关的标准库能力全景，避免重复造轮子。

| 能力 | 所属 | 作用 | 典型场景 |
|---|---|---|---|
| `iter()` / `next()` | 内置函数 | 协议入口 / 取下一个值 | 手动驱动迭代 |
| `itertools.islice` | itertools | 从（无限）流中切片 | 取前 N 条、分批 |
| `itertools.count` | itertools | 无限计数器 | 生成 id 序列 |
| `itertools.chain` | itertools | 串联多个可迭代对象 | 合并多数据源 |
| `itertools.tee` | itertools | 一个流拆成多个 | 多次消费同一生成器 |
| `itertools.accumulate` | itertools | 累积聚合 | 运行统计、前缀和 |
| `itertools.batched`（3.12+） | itertools | 自动分批 | 批量调用 API |
| `collections.abc.Iterable/Iterator` | collections.abc | 类型判定 | 校验接口入参 |
| `contextlib.contextmanager` | contextlib | 用生成器写上下文管理器 | 资源/计时/重试 |
| `async for` / `asyncio` | asyncio | 异步生成器消费 | 流式 LLM 响应 |


In [30]:
# 知识点 20：itertools 常用工具速览
from itertools import islice, count, chain, accumulate

print("count 取前 5 :", list(islice(count(10, 5), 5)))       # 从 10 步长 5
print("chain 串联  :", list(chain([1, 2], "ab", [3])))       # 混合类型串联
print("accumulate  :", list(accumulate([1, 2, 3, 4])))      # 累积和

count 取前 5 : [10, 15, 20, 25, 30]
chain 串联  : [1, 2, 'a', 'b', 3]
accumulate  : [1, 3, 6, 10]


---
# 十五、综合实战：LLM 语料清洗入库管道

**本章解决什么问题**：把前面多个知识点（数据源生成器、过滤、分块、批处理、确定性模拟、统计）组合成一条贴近 LLMOps 业务的完整管道。

## 设计取舍

| 决策 | 取舍说明 |
|---|---|
| 每步拆成独立小生成器 | 单一职责 → 可单测、可任意重组；代价是函数数量多 |
| 全链路惰性 | 百万行语料内存占用恒定；代价是调试时不易"看到中间结果" |
| 分块时按文档边界重置缓冲 | 避免不同文档的行被拼进同一块（上下文污染）；代价是块尾可能不满 |
| 向量化用确定性哈希模拟 | 保证 Notebook 每次执行输出一致（可复现）；真实场景替换为 embedding API 调用 |
| 组装后才消费 | 组装期零开销；一次性管道符合"入库"语义 |


In [31]:
# 综合实战：多文档 -> 清洗 -> 分块 -> 模拟向量化 -> 批量入库统计
import hashlib

# ① 数据源：模拟 3 份"用户问答文档"（真实场景是 DB 分页或大文件逐行）
RAW_DOCS = {
    "faq_v1.txt": [
        "  什么是 LLMOps？  ",
        "",
        "LLMOps 是大模型应用的开发与运维体系。",
        "   ",
        "如何做流式输出？",
        "用生成器逐 token 处理，节省内存。",
    ],
    "faq_v2.txt": [
        "什么是 RAG？",
        "检索增强生成，先检索后回答。",
        "",
        "为什么要分块？",
        "控制上下文长度并提高召回精度。",
    ],
    "faq_v3.txt": [
        "什么是迭代器协议？",
    ],
}

def read_doc_lines(docs):
    # ② 数据源生成器：逐行产出 (文档名, 行号, 行内容)
    for name, lines in docs.items():
        for i, line in enumerate(lines):
            yield name, i, line

def clean(items):
    # ③ 清洗：去首尾空白 + 丢弃空行
    for name, i, line in items:
        stripped = line.strip()
        if stripped:
            yield name, i, stripped

def chunk_rows(items, size=2):
    # ④ 分块：相邻行攒成小块；换文档时先冲刷缓冲，避免跨文档拼接
    buf, current_doc = [], None
    for name, i, text in items:
        if name != current_doc:
            if buf:
                yield buf
            buf, current_doc = [], name
        buf.append((name, i, text))
        if len(buf) == size:
            yield buf
            buf = []
    if buf:
        yield buf

def fake_embedding(text):
    # ⑤ 模拟向量化：确定性哈希（真实场景调用 embedding API）
    digest = hashlib.md5(text.encode("utf-8")).hexdigest()
    return [int(digest[i:i + 2], 16) for i in (0, 2, 4)]   # 取 3 维示意

def ingest(chunks):
    # ⑥ 入库：逐块"写库"并统计（此处只统计，不产生副作用）
    stats = {"chunks": 0, "vectors": 0, "docs": set()}
    for chunk in chunks:
        for name, i, text in chunk:
            vec = fake_embedding(text)
            stats["chunks"] += 1
            stats["vectors"] += len(vec)
            stats["docs"].add(name)
    return stats

# ⑦ 组装管道：此刻零执行（全惰性）
pipeline = chunk_rows(clean(read_doc_lines(RAW_DOCS)))

# ⑧ 消费：逐级拉取，驱动整条管道
result = ingest(pipeline)
print("入库文本块数 :", result["chunks"])
print("生成向量维数 :", result["vectors"])
print("覆盖文档     :", sorted(result["docs"]))

# ⑨ 复用验证：管道是一次性的，如需再次统计必须重建
pipeline2 = chunk_rows(clean(read_doc_lines(RAW_DOCS)))
print("重建后再统计 :", ingest(pipeline2)["chunks"], "（与首次一致，输出可复现）")

入库文本块数 : 9
生成向量维数 : 27
覆盖文档     : ['faq_v1.txt', 'faq_v2.txt', 'faq_v3.txt']
重建后再统计 : 9 （与首次一致，输出可复现）


---
# 十六、常见陷阱汇总

**本章解决什么问题**：一张"陷阱 / 症状 / 解法"总表，每个陷阱都有可运行的复现代码（前面反例单元已覆盖 1–3，这里补全 4–5）。

| 陷阱 | 症状 | 解法 |
|---|---|---|
| 生成器二次遍历 | 第二次得到 `[]`（静默错误） | 重建生成器 / 转列表 / `itertools.tee` |
| 未启动就 `send(值)` | `TypeError: can't send non-None value...` | 先 `next()` 或 `send(None)` |
| 生成器内 `raise StopIteration` | `RuntimeError`（PEP 479） | 用 `return` 结束 |
| 闭包延迟绑定 | 所有 lambda 返回最后一个值 `[2, 2, 2]` | 默认参数立即绑定 `lambda x=x: ...` |
| 忽略资源释放 | `finally` 迟迟不执行 | `with` 管理资源或显式 `close()` |


In [32]:
# 陷阱 4：闭包延迟绑定（lambda / 生成器表达式同理）
funcs = [lambda: x for x in range(3)]           # 【错误写法】共享同一个 x
print("延迟绑定:", [f() for f in funcs])         # [2, 2, 2]

funcs_ok = [lambda x=x: x for x in range(3)]    # 【正确写法】默认参数在定义时求值
print("默认参数:", [f() for f in funcs_ok])     # [0, 1, 2]

延迟绑定: [2, 2, 2]
默认参数: [0, 1, 2]


In [33]:
# 陷阱 5：资源释放 —— finally 不会"自动立即"执行，需要消费完毕或显式 close
def gen():
    try:
        yield 1
        yield 2
    finally:
        print("清理执行")            # ① 只有生成器结束（耗尽/关闭）才执行

print("--- 显式 close ---")
g = gen()
print(next(g))
g.close()                            # ② 立即触发 finally

print("--- 正常耗尽 ---")
g2 = gen()
print(list(g2))                      # ③ 耗尽时 finally 也会执行

--- 显式 close ---
1
清理执行
--- 正常耗尽 ---
清理执行
[1, 2]


---
# 十七、总结

## 概念速查表

| 概念 | 说明 |
|---|---|
| 可迭代对象 | 实现 `__iter__`，能被 `iter()` |
| 迭代器 | 实现 `__iter__` 和 `__next__`，能被 `next()` |
| 生成器 | 用 `yield` 或生成器表达式创建的迭代器 |
| `yield` | 暂停函数，返回值，保存状态 |
| `next()` | 恢复生成器，取下一个值 |
| `send()` | 向生成器发送值，并恢复执行 |
| `throw()` | 在暂停点抛入异常 |
| `close()` | 关闭生成器，触发清理 |
| `yield from` | 委托给子生成器，简化嵌套 |
| 生成器表达式 | 惰性版本的列表推导 |
| `async for` | 遍历异步生成器 |

## 记忆口诀

1. **迭代靠协议**：`__iter__` 加 `__next__`，取完就抛 `StopIteration`；
2. **for 是语法糖**：`iter()` 拿迭代器，`next()` 循环取值；
3. **yield 即暂停**：函数状态原地保存，`next()` 一叫就醒；
4. **生成器一次性**：遍历完就是空的，要用就重建或转列表；
5. **send 前先启动**：首次必须 `next()` 或 `send(None)`；
6. **大流走管道**：小生成器逐级串联，惰性拉取省内存。

## 核心一句话

> 迭代器是协议，生成器是实现这个协议的便捷方式；生成器的本质是"可暂停、可恢复、可惰性求值"的函数。


In [34]:
# ============ 可复用代码模板 ============

# 模板 1：自定义可迭代对象（借助生成器函数省掉手写迭代器类）
class MyIterable:
    def __init__(self, data):
        self._data = data
    def __iter__(self):                 # 每次遍历都创建新的迭代器 → 可重复遍历
        for item in self._data:
            yield item

print(list(MyIterable([1, 2, 3])))
print(list(MyIterable([1, 2, 3])))       # 两次结果一致

# 模板 2：数据管道骨架（源 -> 过滤 -> 转换 -> 消费）
def source(items):
    yield from items                    # 数据源（DB 分页 / 文件行 / API 流）

def transform(items):
    for x in items:
        yield x * 2                     # 逐项转换

def consume(items):
    return list(items)                  # 最终消费者（聚合 / 入库 / 打印）

print(consume(transform(source([1, 2, 3]))))

[1, 2, 3]
[1, 2, 3]
[2, 4, 6]


---
# 十八、自测练习（先自己写，再看参考实现）

1. 实现一个**斐波那契无限生成器**，并用 `itertools.islice` 取前 10 项；
2. 用 `yield from` 实现 `flatten`，把任意深度嵌套的列表展平成一维；
3. 实现一个 `cycle(seq)` 无限循环生成器，取 `cycle("ab")` 的前 7 项。


In [35]:
# ============ 参考实现 ============
from itertools import islice

# 练习 1：斐波那契无限生成器
def fib():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b            # 元组解包：一步完成状态推进
print("fib 前 10 项:", list(islice(fib(), 10)))

# 练习 2：yield from 展平嵌套列表
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)   # 子列表整体委托给递归调用
        else:
            yield item
print("展平:", list(flatten([1, [2, [3, 4], 5], 6])))

# 练习 3：无限循环 cycle
def cycle(seq):
    while True:
        yield from seq             # 一轮结束再来一轮
print("cycle 前 7 项:", list(islice(cycle("ab"), 7)))

fib 前 10 项: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
展平: [1, 2, 3, 4, 5, 6]
cycle 前 7 项: ['a', 'b', 'a', 'b', 'a', 'b', 'a']


---
## 勘误说明

经逐条验证，源文档内容在技术事实上未发现错误，本 Notebook 全部结论均以实际执行输出佐证。两处工程化补充：

1. 源文档异步示例使用 `asyncio.run(main())`；在 Notebook 内核（已有事件循环管理）中更稳妥的写法是 `new_event_loop() + run_until_complete()`，本 Notebook 采用后者（见第十一章知识点 19）。
2. 源文档闭包延迟绑定的修正在工程上首选**默认参数立即绑定**（`lambda x=x: x`），已在陷阱 4 中给出完整对比。
